In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
# load acs5_2023_tract_CA
df = pd.read_csv('../data/raw/acs5_2023_tract_CA.csv', dtype={'state': str, 'county': str, 'tract': str})

# create GEO ID column
df['GEOID'] = df['state'] + df['county'] + df['tract']

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9129 entries, 0 to 9128
Data columns (total 44 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   NAME         9129 non-null   object 
 1   B19013_001E  9129 non-null   int64  
 2   B17001_001E  9129 non-null   int64  
 3   B17001_002E  9129 non-null   int64  
 4   B25070_001E  9129 non-null   int64  
 5   B25070_002E  9129 non-null   int64  
 6   B25070_003E  9129 non-null   int64  
 7   B25070_004E  9129 non-null   int64  
 8   B25070_005E  9129 non-null   int64  
 9   B25070_006E  9129 non-null   int64  
 10  B25070_007E  9129 non-null   int64  
 11  B25070_008E  9129 non-null   int64  
 12  B25070_009E  9129 non-null   int64  
 13  B25070_010E  9129 non-null   int64  
 14  B15003_001E  9129 non-null   int64  
 15  B15003_002E  9129 non-null   int64  
 16  B15003_003E  9129 non-null   int64  
 17  B15003_004E  9129 non-null   int64  
 18  B15003_005E  9129 non-null   int64  
 19  B15003

In [3]:
# verify 11 characters per GEOID
df['GEOID'].str.len().value_counts()

GEOID
11    9129
Name: count, dtype: int64

In [4]:
# Check if GEOID's match all instances of tract_fips
print((df['GEOID'] == df['tract_fips'].astype(str)).all())

False


In [5]:
# check tract_fips data type and instance length
print(df['tract_fips'].dtype)
print(df['tract_fips'].astype(str).str.len().value_counts())

int64
tract_fips
10    9129
Name: count, dtype: int64


In [6]:
# verify that GEOID is the same as string converted tract_fips
print((df['GEOID'].str[1:] == df['tract_fips'].astype(str)).all())

True


In [7]:
# drop tract_fips as GEOID replaces it
df = df.drop(columns=['tract_fips'])

In [8]:
print(df.columns)

Index(['NAME', 'B19013_001E', 'B17001_001E', 'B17001_002E', 'B25070_001E',
       'B25070_002E', 'B25070_003E', 'B25070_004E', 'B25070_005E',
       'B25070_006E', 'B25070_007E', 'B25070_008E', 'B25070_009E',
       'B25070_010E', 'B15003_001E', 'B15003_002E', 'B15003_003E',
       'B15003_004E', 'B15003_005E', 'B15003_006E', 'B15003_007E',
       'B15003_008E', 'B15003_009E', 'B15003_010E', 'B15003_011E',
       'B15003_012E', 'B15003_013E', 'B15003_014E', 'B15003_015E',
       'B15003_016E', 'B23025_003E', 'B23025_005E', 'B25002_001E',
       'B25002_003E', 'B25003_001E', 'B25003_003E', 'B25064_001E',
       'B01003_001E', 'B01002_001E', 'state', 'county', 'tract', 'GEOID'],
      dtype='object')


In [9]:
rename_map = {
    # Income / poverty
    "B19013_001E": "median_hh_income",
    "B17001_001E": "poverty_status_total",
    "B17001_002E": "pop_below_poverty",
    # Rent burden
    "B25070_001E": "renter_hh_total",
    "B25070_002E": "rent_burden_lt10",
    "B25070_003E": "rent_burden_10_15",
    "B25070_004E": "rent_burden_15_20",
    "B25070_005E": "rent_burden_20_25",
    "B25070_006E": "rent_burden_25_30",
    "B25070_007E": "rent_burden_30_35",
    "B25070_008E": "rent_burden_35_40",
    "B25070_009E": "rent_burden_40_50",
    "B25070_010E": "rent_burden_50plus",
    # Education (adults 25+)
    "B15003_001E": "edu_total_25plus",
    "B15003_002E": "edu_no_school",
    "B15003_003E": "edu_nursery",
    "B15003_004E": "edu_kindergarten",
    **{f"B15003_{i:03d}E": f"edu_grade_{i - 4}" for i in range(5, 16)},
    "B15003_016E": "edu_12th_no_diploma",
    # Employment
    "B23025_003E": "labor_force",
    "B23025_005E": "unemployed",
    # Housing
    "B25002_001E": "housing_units_total",
    "B25002_003E": "housing_units_vacant",
    "B25003_001E": "occupied_units_total",
    "B25003_003E": "renter_occupied_units",
    "B25064_001E": "median_gross_rent",
    # Population
    "B01003_001E": "total_pop",
    "B01002_001E": "median_age",
}

df = df.rename(columns=rename_map)
print(len(rename_map))
df.info()

38
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9129 entries, 0 to 9128
Data columns (total 43 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   NAME                   9129 non-null   object 
 1   median_hh_income       9129 non-null   int64  
 2   poverty_status_total   9129 non-null   int64  
 3   pop_below_poverty      9129 non-null   int64  
 4   renter_hh_total        9129 non-null   int64  
 5   rent_burden_lt10       9129 non-null   int64  
 6   rent_burden_10_15      9129 non-null   int64  
 7   rent_burden_15_20      9129 non-null   int64  
 8   rent_burden_20_25      9129 non-null   int64  
 9   rent_burden_25_30      9129 non-null   int64  
 10  rent_burden_30_35      9129 non-null   int64  
 11  rent_burden_35_40      9129 non-null   int64  
 12  rent_burden_40_50      9129 non-null   int64  
 13  rent_burden_50plus     9129 non-null   int64  
 14  edu_total_25plus       9129 non-null   int64  
 15  e

In [10]:
df.head()

,NAME,median_hh_income,poverty_status_total,pop_below_poverty,renter_hh_total,rent_burden_lt10,rent_burden_10_15,rent_burden_15_20,rent_burden_20_25,rent_burden_25_30,...,housing_units_vacant,occupied_units_total,renter_occupied_units,median_gross_rent,total_pop,median_age,state,county,tract,GEOID
0,Census Tract 4001; Alameda County; California,250001,3075,134,109,8,8,75,0,10,...,139,1316,109,3501,3094,51.4,06,001,400100,06001400100
1,Census Tract 4002; Alameda County; California,225880,2093,164,355,46,41,84,42,100,...,32,861,355,2949,2093,45.0,06,001,400200,06001400200
2,Census Tract 4003; Alameda County; California,157731,5707,310,1632,114,165,235,147,159,...,184,2713,1632,2246,5727,39.6,06,001,400300,06001400300
3,Census Tract 4004; Alameda County; California,159612,4351,343,827,124,126,154,152,20,...,193,1803,827,2485,4395,37.6,06,001,400400,06001400400
4,Census Tract 4005; Alameda County; California,96250,3816,397,871,25,112,149,67,111,...,132,1655,871,2154,3822,35.6,06,001,400500,06001400500


In [11]:
# Check for duplicate tracts 
print(df['GEOID'].duplicated().sum())

0


In [12]:
# check for census negative placeholder values
df.min()

NAME                     Census Tract 1.01; Amador County; California
median_hh_income                                           -666666666
poverty_status_total                                                0
pop_below_poverty                                                   0
renter_hh_total                                                     0
rent_burden_lt10                                                    0
rent_burden_10_15                                                   0
rent_burden_15_20                                                   0
rent_burden_20_25                                                   0
rent_burden_25_30                                                   0
rent_burden_30_35                                                   0
rent_burden_35_40                                                   0
rent_burden_40_50                                                   0
rent_burden_50plus                                                  0
edu_total_25plus    

In [13]:
median_cols = ['median_hh_income', 'median_gross_rent', 'median_age']
df[median_cols] = df[median_cols].replace(-666666666, np.nan)

print(df[median_cols].isna().sum())
print(df[median_cols].min())

median_hh_income     126
median_gross_rent    301
median_age            51
dtype: int64
median_hh_income     7000.0
median_gross_rent     323.0
median_age             10.8
dtype: float64


In [14]:
# investigate the median age of 10.8 finding
df[df['median_age'] < 18].T

,3848,3866,6576,7826
NAME,Census Tract 9800.08; Los Angeles County; Cali...,Census Tract 9800.26; Los Angeles County; Cali...,Census Tract 95.11; San Diego County; California,Census Tract 9805; Santa Barbara County; Calif...
median_hh_income,NaN,NaN,69087.0,NaN
poverty_status_total,73,0,4283,26
pop_below_poverty,64,0,360,0
renter_hh_total,18,0,942,5
rent_burden_lt10,0,0,5,0
rent_burden_10_15,5,0,0,0
rent_burden_15_20,0,0,4,0
rent_burden_20_25,0,0,25,0
rent_burden_25_30,0,0,42,0


In [15]:
# check number of top-coded median household incomes (Census cap of 250001)
(df['median_hh_income'] == 250001).sum()

140

In [16]:
#flag tracts where median household income is top-coded at the Census cap
df['income_topcoded'] = df['median_hh_income'] == 250001
print(df['income_topcoded'].sum())

140


In [17]:
# tract 95.11 SD: possible military housing, unverified

# save to csv
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/acs5_2023_tract_CA_clean.csv', index=False)

In [18]:
places_df = pd.read_csv('../data/raw/places_tract_CA.csv')
print(places_df.columns)

Index(['year', 'stateabbr', 'statedesc', 'countyname', 'countyfips',
       'locationname', 'datasource', 'category', 'measure', 'data_value_unit',
       'data_value_type', 'data_value', 'low_confidence_limit',
       'high_confidence_limit', 'totalpopulation', 'totalpop18plus',
       'locationid', 'categoryid', 'measureid', 'datavaluetypeid',
       'short_question_text'],
      dtype='object')
